# Particle filters — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def normalize(v):
    v=np.asarray(v,dtype=float); return v/v.sum()
def softmax(v):
    v=np.asarray(v,dtype=float); e=np.exp(v-v.max()); return e/e.sum()
def norm_pdf(x,mu,var):
    return np.exp(-0.5*(np.asarray(x)-mu)**2/var)/np.sqrt(2*np.pi*var)
def beta_pdf_grid(x,a,b):
    B=math.gamma(a)*math.gamma(b)/math.gamma(a+b)
    return (np.asarray(x)**(a-1))*((1-np.asarray(x))**(b-1))/B
print('setup ok')

## Approximate a changing posterior with weighted simulated particles
Particle filters represent belief by particles. These examples propagate particles, weight by likelihood, normalize, resample, and show degeneracy through effective sample size.

In [ ]:
# 1) particles predict through dynamics
rng=np.random.default_rng(0); particles=rng.normal(0,1,200); pred=particles+1+rng.normal(0,0.2,200)
plt.figure(figsize=(6,3)); plt.hist(pred,bins=25); plt.title('prediction particles after motion')
assert abs(pred.mean()-0.9975625196832745)<1e-12

In [ ]:
# 2) observation likelihood creates weights
z=1.2; R=0.3; w=norm_pdf(pred,z,R); w=w/w.sum()
plt.figure(figsize=(6,3)); plt.scatter(pred,w,s=12); plt.title('particles near observation get weight')
assert abs(w.sum()-1)<1e-12

In [ ]:
# 3) weighted posterior mean summarizes belief
m=(w*pred).sum(); plt.figure(figsize=(6,3)); plt.axvline(m,c='r'); plt.hist(pred,bins=25,weights=w); plt.title(f'weighted mean={m:.3f}')
assert 1.0<m<1.3

In [ ]:
# 4) resampling duplicates high-weight particles
idx=rng.choice(len(pred),size=len(pred),p=w); res=pred[idx]
plt.figure(figsize=(6,3)); plt.hist(res,bins=25); plt.title('resampled cloud follows posterior')
assert abs(res.mean()-m)<0.15

In [ ]:
# 5) effective sample size detects degeneracy
ess=1/np.sum(w*w); plt.figure(figsize=(6,3)); plt.bar(['ESS','N'],[ess,len(w)]); plt.title('weight concentration lowers ESS')
assert ess<len(w) and ess>50